In [ ]:
import os
import re
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def extract_text_and_params_from_pdf(pdf_path="/content/drive/MyDrive/AL_2026-01/AL_G001/Taller_2/document.pdf"):
    """
    Intenta extraer texto de PDF y detectar parámetros tipo pXd (ej: p1a, p2b, p3e).
    Devuelve (pregunta_texto, sorted_list_of_param_names).
    Si falla la extracción, devuelve un placeholder y una lista de parámetros heurística.
    """
    texto = ""
    pregunta = ""
    try:
        from PyPDF2 import PdfReader
    except Exception:
        # PyPDF2 no disponible: devolver placeholder y parámetros comunes
        placeholder = (
            "EXTRACCION_FALLIDA: No se pudo extraer el PDF automáticamente. "
            "Se supone que el enunciado contiene parámetros tipo p1a..p1i, p2a..p2c, p3a..p3e."
        )
        # devolver parámetros típicos detectados previamente en el PDF proporcionado por el usuario
        guessed = ["p1a","p1b","p1c","p1d","p1e","p1f","p1g","p1h","p1i",
                   "p2a","p2b","p2c",
                   "p3a","p3b","p3c","p3d","p3e"]
        return placeholder, guessed

    # Si PyPDF2 está disponible, intentar leer
    if not os.path.exists(pdf_path):
        placeholder = f"EXTRACCION_FALLIDA: archivo no encontrado en '{pdf_path}'."
        guessed = ["p1a","p1b","p1c","p1d","p1e","p1f","p1g","p1h","p1i",
                   "p2a","p2b","p2c",
                   "p3a","p3b","p3c","p3d","p3e"]
        return placeholder, guessed

    try:
        reader = PdfReader(pdf_path)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                texto += "\n" + page_text
    except Exception as e:
        placeholder = f"EXTRACCION_FALLIDA: error leyendo PDF: {e}"
        guessed = ["p1a","p1b","p1c","p1d","p1e","p1f","p1g","p1h","p1i",
                   "p2a","p2b","p2c",
                   "p3a","p3b","p3c","p3d","p3e"]
        return placeholder, guessed

    # Normalizar texto y buscar parámetros con regex
    # patrón: letra 'p' seguida de 1+ dígitos y 1+ letras (ej: p1a, p10b)
    found = re.findall(r'\bp\d+[a-zA-Z]\b', texto)
    found = [f.lower() for f in found]
    params = sorted(set(found), key=lambda s: (int(re.findall(r'\d+', s)[0]) if re.findall(r'\d+', s) else 0, s))
    # intentar extraer el bloque de la primera pregunta como 'pregunta'
    # heurística: buscar "1." o "1." seguido de "Considere"... tomar hasta la siguiente pregunta "2."
    m = re.search(r'1\..*?(?=2\.)', texto, flags=re.S | re.I)
    if m:
        pregunta = m.group(0).strip()
    else:
        # fallback: tomar las primeras 400 caracteres
        pregunta = texto.strip()[:400] + ("..." if len(texto.strip()) > 400 else "")

    # Si no detectó parámetros, usar parámetros típicos (como fallback)
    if not params:
        params = ["p1a","p1b","p1c","p1d","p1e","p1f","p1g","p1h","p1i",
                  "p2a","p2b","p2c",
                  "p3a","p3b","p3c","p3d","p3e"]
    return pregunta, params

def generate_nonzero_integer_vector(length, low=-10, high=10, rng=None):
    """
    Genera un vector numpy de 'length' enteros en [low, high] excluyendo 0.
    """
    if rng is None:
        rng = np.random.default_rng()
    if low >= high:
        raise ValueError("low debe ser menor que high")
    out = np.empty(length, dtype=int)
    i = 0
    while i < length:
        candidate = int(rng.integers(low, high + 1))
        if candidate != 0:
            out[i] = candidate
            i += 1
    return out

def build_matrices_from_params(params_row):
    """
    Dado un diccionario params_row con claves p1a..p3e, construye:
      - M1 (3x3) para P1 (columnas: [p1a,p1b,p1c], [p1d,p1e,p1f], [p1g,p1h,p1i])
      - V (3x3) base para P2 (const anteponer): V tiene columnas v1,v2,v3 según problema
      - w (3,) vector para P2: [p2a,p2b,p2c]
      - B (4x4) para P3 según disposición en el enunciado
    Devuelve (M1, V, w, B). Si faltan claves, lanza KeyError.
    """
    # M1
    M1 = np.column_stack([
        [params_row["p1a"], params_row["p1b"], params_row["p1c"]],
        [params_row["p1d"], params_row["p1e"], params_row["p1f"]],
        [params_row["p1g"], params_row["p1h"], params_row["p1i"]],
    ]).astype(float)  # cada columna será reorganizada a (3,) — arreglamos abajo
    # NOTE: Above column_stack produced a (3,3) where columns are already correct if we instead do:
    # Let's construct correctly: each vector is a column [p1a, p1b, p1c]^T etc.
    M1 = np.column_stack([
        [params_row["p1a"], params_row["p1b"], params_row["p1c"]],
        [params_row["p1d"], params_row["p1e"], params_row["p1f"]],
        [params_row["p1g"], params_row["p1h"], params_row["p1i"]],
    ]).T.astype(float)  # transpose to get columns as intended
    # P2: basis V columns v1=[1,0,0], v2=[1,1,0], v3=[1,1,1]
    V = np.array([[1,1,1],
                  [0,1,1],
                  [0,0,1]], dtype=float)
    w = np.array([params_row["p2a"], params_row["p2b"], params_row["p2c"]], dtype=float)
    # P3: B matrix per enunciado:
    # B = [[p3b, p3c, p3d, p3e],
    #      [0,   -1,  1,   2],
    #      [5,    6, -1,   1],
    #      [0,    0, p3a,   0]]
    B = np.array([
        [params_row["p3b"], params_row["p3c"], params_row["p3d"], params_row["p3e"]],
        [0, -1, 1, 2],
        [5, 6, -1, 1],
        [0, 0, params_row["p3a"], 0]
    ], dtype=float)
    return M1, V, w, B

def evaluate_problemset(n=20, low=-10, high=10, seed=None, ensure_nondegenerate=False, max_attempts=200, pdf_path="/content/drive/MyDrive/AL_2026-01/AL_G001/Taller_2/document.pdf"):
    """
    Genera n versiones del conjunto de ejercicios y evalúa resultados.
    Parámetros:
      - n: número de muestras/versiones
      - low, high: rango inclusivo para generar enteros (0 es excluido)
      - seed: semilla para reproducibilidad (None = aleatorio)
      - ensure_nondegenerate: si True, re-genera muestras individuales si P1 o P3 resultan en determinante 0
      - max_attempts: número máximo de reintentos por muestra para evitar degeneraciones
      - pdf_path: ruta al PDF para extraer 'pregunta' y lista de parámetros
    Retorna:
      - pregunta (str)
      - df (pandas.DataFrame) con columnas: [params..., p1_det, p1_independent, c1, c2, c3, B_det]
    """
    rng = np.random.default_rng(seed)

    pregunta, detected_params = extract_text_and_params_from_pdf(pdf_path)

    # Asegurarnos de que los parámetros esperados para cada problema estén presentes; si no, agregamos los típicos
    expected = ["p1a","p1b","p1c","p1d","p1e","p1f","p1g","p1h","p1i",
                "p2a","p2b","p2c",
                "p3a","p3b","p3c","p3d","p3e"]
    for p in expected:
        if p not in detected_params:
            detected_params.append(p)
    # Ordenar por número/letra para consistencia de columnas
    detected_params = sorted(set(detected_params), key=lambda s: (int(re.findall(r'\d+', s)[0]) if re.findall(r'\d+', s) else 0, s))

    # Generar matriz (len(detected_params) x n) de parámetros no cero
    params_matrix = {}
    for pname in detected_params:
        params_matrix[pname] = generate_nonzero_integer_vector(n, low=low, high=high, rng=rng)

    # Preparar arrays de resultados
    results = {p: params_matrix[p].copy() for p in detected_params}
    p1_det = np.zeros(n, dtype=float)
    p1_indep = np.zeros(n, dtype=bool)
    c1 = np.zeros(n, dtype=float)
    c2 = np.zeros(n, dtype=float)
    c3 = np.zeros(n, dtype=float)
    B_det = np.zeros(n, dtype=float)

    # Para cada muestra, evaluar P1, P2, P3
    for i in range(n):
        attempts = 0
        while True:
            # Construir un dict con parámetros de la muestra i
            sample = {p: int(results[p][i]) for p in detected_params}
            try:
                M1, V, w, B = build_matrices_from_params(sample)
            except KeyError as e:
                raise KeyError(f"Falta parámetro necesario en el PDF/extracción: {e}")

            # Evaluar determinante P1
            det1 = float(np.linalg.det(M1))
            # Tratar valores muy cercanos a enteros
            det1_rounded = float(np.rint(det1))
            if abs(det1 - det1_rounded) < 1e-8:
                det1 = det1_rounded
            is_indep = not np.isclose(det1, 0.0, atol=1e-9)

            # Evaluar P2: resolver V * c = w (V es triangular superior con det 1, invertible)
            try:
                c = np.linalg.solve(V, w)
            except np.linalg.LinAlgError:
                # Esto no debería ocurrir con la V dada, pero lo manejamos
                c = np.array([np.nan, np.nan, np.nan], dtype=float)

            # Evaluar P3: determinante de B
            detB = float(np.linalg.det(B))
            detB_rounded = float(np.rint(detB))
            if abs(detB - detB_rounded) < 1e-8:
                detB = detB_rounded
            is_B_nonzero = not np.isclose(detB, 0.0, atol=1e-9)

            # Si se piden no degeneraciones, re-generar la muestra si P1 o P3 son degeneradas (det == 0)
            if ensure_nondegenerate and (not is_indep or not is_B_nonzero):
                attempts += 1
                if attempts >= max_attempts:
                    # no pudimos encontrar una versión no degenerada para esta muestra; romper y aceptar la degeneración
                    break
                # re-generar todos los parámetros usados por P1 y P3 para la muestra i (manteniendo valores de otros parámetros)
                for pname in detected_params:
                    # Regenerar sólo si pertenece a p1* o p3* (no es necesario regenerar p2*, pero es seguro hacerlo también)
                    if pname.startswith("p1") or pname.startswith("p3") or pname.startswith("p2"):
                        results[pname][i] = generate_nonzero_integer_vector(1, low=low, high=high, rng=rng)[0]
                # volver a intentar
                continue
            # Si no pedimos asegurar no degeneración, o si pasamos las comprobaciones/limite de intentos, guardamos resultados
            p1_det[i] = det1
            p1_indep[i] = is_indep
            c1[i], c2[i], c3[i] = c
            B_det[i] = detB
            break  # salir del while para pasar a la siguiente muestra

    # Construir DataFrame final
    df_dict = {p: results[p] for p in detected_params}
    df_dict.update({
        "p1_det": p1_det,
        "p1_independent": p1_indep,
        "c1": c1,
        "c2": c2,
        "c3": c3,
        "B_det": B_det
    })
    df = pd.DataFrame(df_dict)

    # Validaciones finales
    # - Asegurar que no hay ceros en parámetros (lo garantizamos en generación)
    if (df[[p for p in detected_params]] == 0).any().any():
        raise RuntimeError("Se detectó un 0 en los parámetros generados (no debería ocurrir).")

    return pregunta, detected_params, df

In [ ]:
N = 100                 # número de versiones que queremos generar
LOW, HIGH = -12, 12     # rango para enteros; 0 será excluido por la función generadora
SEED = 123             # semilla opcional para reproducibilidad
ENSURE_NONDEG = True    # intentar evitar matrices singulares (P1 y P3) regenerando muestras
PDF_PATH = "/content/drive/MyDrive/AL_2026-01/AL_G001/Taller_2/document.pdf"

pregunta_texto, params_list, dataframe = evaluate_problemset(
        n=N, low=LOW, high=HIGH, seed=SEED,
        ensure_nondegenerate=ENSURE_NONDEG, max_attempts=500,
        pdf_path=PDF_PATH)

In [ ]:
ruta_L = '/content/drive/MyDrive/AL_2026-01/AL_G001/Libro1.xlsx'
L = pd.read_excel(ruta_L)
L['name'] = L['ApellidoDeEstudiante'] + ' ' + L['NombreDeEstudiante']
L = L.drop(columns=['ApellidoDeEstudiante', 'NombreDeEstudiante'])
Preguntas = pd.concat([L, dataframe.iloc[:len(L)]], axis=1)
Preguntas = Preguntas.reset_index(drop=False)  # Mueve el índice a una columna llamada "index"

Preguntas =Preguntas.rename(columns={'index': 'num','Identificacion':'id','EmailInstitucional':'Correo'})  # Renombra esa columna
Preguntas['num'] = Preguntas['num'] + 1  # Incrementa el índice en 1
ruta_G = '/content/drive/MyDrive/AL_2026-01/AL_G001/Taller_2/Taller_2.xlsx'
Preguntas.to_excel(ruta_G ,index=False)

In [ ]:
!sudo dpkg --configure -a
!apt-get update  # Optional: Update package lists
!apt-get install -y texlive-latex-extra

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
texlive-latex-extra is already the newest version (2021.20220204-1).
0 upgraded, 0 newly insta

In [ ]:
p_variables = Preguntas.columns.to_list()
p_variables = ['num','name','id'] + p_variables


import os
import re

# Assuming the .tex file is in the same directory as the notebook
tex_file_path = '/content/drive/MyDrive/AL_2026-01/AL_G001/Taller_2/document.tex'  # Replace with the actual path

# Read the .tex file
with open(tex_file_path, 'r') as f:
    tex_content = f.read()


def replace_values_in_tex(tex_content, df, n=2):
    """Replaces placeholders in the .tex file with DataFrame values.

    Args:
        tex_content: The content of the .tex file as a string.
        df: The pandas DataFrame containing the values to substitute.
        n: The number of .pdf files to generate.

    Returns:
        A list of modified .tex file content, one for each .pdf file.
    """
    modified_tex_contents = []
    for i in range(min(n, len(df))):
        new_tex = tex_content
        for col in p_variables:
           replacement_string = str(df[col].iloc[i]).replace('\\', '\\\\')
           new_tex = re.sub(r'param-' + col, replacement_string, new_tex)
        modified_tex_contents.append(new_tex)

    return modified_tex_contents


modified_tex_contents = replace_values_in_tex(tex_content, Preguntas, n=len(Preguntas))

# Generate the .tex files and compile them to .pdf
for i, modified_tex in enumerate(modified_tex_contents):
    temp_tex_file = f'/content/drive/MyDrive/AL_2026-01/AL_G001/Taller_2/temp_{i+1}.tex'
    with open(temp_tex_file, 'w') as f:
        f.write(modified_tex)

    # Compile the .tex file to .pdf using pdflatex
    !pdflatex -output-directory="/content/drive/MyDrive/AL_2026-01/AL_G001/Taller_2/" "{temp_tex_file}"

    # Clean up the temporary .tex file
    os.remove(temp_tex_file)


# Delete .log and .aux files in the specified directory
directory_to_clean = '/content/drive/MyDrive/AL_2026-01/AL_G001/Taller_2/'
for filename in os.listdir(directory_to_clean):
    if filename.endswith(('.log', '.aux')):
        file_path = os.path.join(directory_to_clean, filename)
        try:
            os.remove(file_path)
            print(f"Deleted file: {file_path}")
        except OSError as e:
            print(f"Error deleting file {file_path}: {e}")

Se han truncado las últimas 5000 líneas del flujo de salida.
(/usr/share/texlive/texmf-dist/tex/generic/pgfplots/pgfplots.paths.code.tex)
(/usr/share/texlive/texmf-dist/tex/generic/pgf/frontendlayer/tikz/libraries/tik
zlibrarydecorations.code.tex
(/usr/share/texlive/texmf-dist/tex/generic/pgf/modules/pgfmoduledecorations.cod
e.tex))
(/usr/share/texlive/texmf-dist/tex/generic/pgf/frontendlayer/tikz/libraries/tik
zlibrarydecorations.pathmorphing.code.tex
(/usr/share/texlive/texmf-dist/tex/generic/pgf/libraries/decorations/pgflibrary
decorations.pathmorphing.code.tex))
(/usr/share/texlive/texmf-dist/tex/generic/pgf/frontendlayer/tikz/libraries/tik
zlibrarydecorations.pathreplacing.code.tex
(/usr/share/texlive/texmf-dist/tex/generic/pgf/libraries/decorations/pgflibrary
decorations.pathreplacing.code.tex))
(/usr/share/texlive/texmf-dist/tex/generic/pgfplots/libs/tikzlibrarypgfplots.co
ntourlua.code.tex))
(/usr/share/texlive/texmf-dist/tex/generic/pgf/frontendlayer/tikz/libraries/tik
zlibrar

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

def send_email_with_attachment(sender_email, sender_password, receiver_email, subject, body, filename):
    """
    Envía un correo electrónico con un archivo adjunto utilizando la biblioteca smtplib.

    Args:
        sender_email (str): La dirección de correo del remitente.
        sender_password (str): La contraseña del remitente.
        receiver_email (str): La dirección de correo del destinatario.
        subject (str): El asunto del correo.
        body (str): El cuerpo del correo en texto plano.
        filename (str): El nombre del archivo a adjuntar. Se espera que esté en la carpeta "Colab Notebooks" de Google Drive.
    """
    # Crear el objeto MIMEMultipart
    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg['Subject'] = subject

    # Adjuntar el cuerpo del correo
    msg.attach(MIMEText(body, 'plain'))

    # Adjuntar el archivo
    attachment_path = os.path.join("/content/drive/MyDrive/AL_2026-01/AL_G001/Taller_2/", filename)

    try:
        with open(attachment_path, "rb") as attachment:
            part = MIMEBase("application", "octet-stream")
            part.set_payload(attachment.read())
        encoders.encode_base64(part)
        part.add_header(
            "Content-Disposition",
            f"attachment; filename= {filename}",
        )
        msg.attach(part)

        # Conectar al servidor SMTP de Gmail
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls() # Iniciar cifrado TLS
        server.login(sender_email, sender_password)

        # Enviar el correo
        server.sendmail(sender_email, receiver_email, msg.as_string())

        # Cerrar la conexión
        server.quit()

        print("Correo enviado a {} con éxito.".format(receiver_email))

    except FileNotFoundError:
        print(f"Error: El archivo '{filename}' no se encontró en la ruta especificada.")
    except smtplib.SMTPAuthenticationError:
        print("Error: Falló la autenticación. Verifica tu correo y contraseña.")
        print("Si tienes la autenticación de dos factores activada, es posible que necesites una contraseña de aplicación.")
    except Exception as e:
        print(f"Error al enviar el correo: {e}")

In [ ]:
from google.colab import userdata
sender_email = "wilmar.gonzalez@pascualbravo.edu.co" # Reemplaza con tu correo
sender_password = userdata.get("sender_password")# Reemplaza con tu contraseña o utiliza una contraseña de aplicación si tienes 2FA activado
subject = "Taller 2"
body = '''Buenas noches {}, espero que se encuentre muy bien.

En el presente le anexo el simulacro uno, por favor verifique que los datos allí mostrados coinciden con
los suyos, una vez haya revisado esto proceda a solucionar dicha actividad y al terminar por favor
responda el siguiente formulario https://forms.gle/PoLsH4GHsBc4goHJ7, en este únicamente se agregan los
valores numéricos obtenidos en cada ejercicio.

Éxitos y feliz noche.

Cordialmente,

Wilmar A. González Medina'''

for i in range(len(Preguntas)):
  receiver_email = Preguntas.loc[i,'Correo'] # Reemplaza con el correo del destinatario
  filename = "temp_{}.pdf".format(i+1)  # Reemplaza con el nombre de tu archivo PDF
  bodi = body.format(Preguntas.loc[i,'name'])
  send_email_with_attachment(sender_email, sender_password, receiver_email, subject, bodi, filename)

Correo enviado a s.zuleta1205@pascualbravo.edu.co con éxito.
Correo enviado a samuel.arias776@pascualbravo.edu.co con éxito.
Correo enviado a ashly.becerra536@pascualbravo.edu.co con éxito.
Correo enviado a julian.betancur562@pascualbravo.edu.co con éxito.
Correo enviado a ivan.bolanos872@pascualbravo.edu.co con éxito.
Correo enviado a deiman.buitrago240@pascualbravo.edu.co con éxito.
Correo enviado a maria.cantillo180@pascualbravo.edu.co con éxito.
Correo enviado a brandon.cartagena079@pascualbravo.edu.co con éxito.
Correo enviado a miguel.castrillon313@pascualbravo.edu.co con éxito.
Correo enviado a johana.catano278@pascualbravo.edu.co con éxito.
Correo enviado a jagler.caviedes566@pascualbravo.edu.co con éxito.
Correo enviado a deimar.copete258@pascualbravo.edu.co con éxito.
Correo enviado a linda.cordoba461@pascualbravo.edu.co con éxito.
Correo enviado a brayan.cordoba260@pascualbravo.edu.co con éxito.
Correo enviado a paula.coronado356@pascualbravo.edu.co con éxito.
Correo enviado